# Chapter 5 — Durable, Tree-Structured Sessions

This chapter turns application-facing run control into resumable local Session state without overstating crash recovery.

## Goal and Previous Limitation

Chapter 4 owns queues, cancellation, and one active Run, but its history exists only inside one `AgentRuntime`. Process exit loses complete model and Tool outcomes, and callers have no explicit continuation or historical fork. We begin from the immutable Chapter 4 Checkpoint and add durability only at Settled Boundaries.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_4 = ROOT / 'course' / 'checkpoints' / 'ch04'
sys.path.insert(0, str(CHAPTER_4 / 'src'))
import agent_harness as chapter4

assert hasattr(chapter4, 'AgentSession')
assert not hasattr(chapter4, 'JSONLSessionStore')
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]
sys.path.remove(str(CHAPTER_4 / 'src'))

## Conceptual Model

`AgentSession` becomes the durable façade; `AgentRuntime` remains the in-memory engine. A small `SessionStore` interface—`create`, `read`, and `writer`—has two real adapters: `MemorySessionStore` and `JSONLSessionStore`. A `SessionEntry` stores one structural settlement and a parent identifier. Following parents materializes one branch while all other entries remain navigable.

The Runtime calls a Session-owned settlement sink synchronously. A plain message may settle alone; an assistant Tool Call and all ordered Tool results settle together. This seam gives persistence leverage without putting files, locks, or schema decoding inside the model loop.

## Minimal Execution

The next Export Cells add the persistence module and replace the evolved Runtime, Session, and package interface. Each cell owns one complete module.

In [ ]:
PERSISTENCE_SOURCE = '"""Tree-structured Session persistence at settled boundaries."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import BinaryIO, Protocol, TypeAlias\nfrom uuid import uuid4\n\nfrom .model import AgentMessage, Role, TextContent, ToolCallContent\nfrom .tools import (\n    CompleteOutputKind,\n    CompleteOutputReference,\n    ToolErrorCode,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nIdFactory: TypeAlias = Callable[[], str]\n\n\n@dataclass(frozen=True, slots=True)\nclass SchemaVersion:\n    major: int\n    minor: int = 0\n\n\nSESSION_SCHEMA_VERSION = SchemaVersion(1, 0)\n_SESSION_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}\\Z")\n\n\nclass RecoveryCode(str, Enum):\n    INCOMPLETE_FINAL_RECORD = "incomplete_final_record"\n\n\n@dataclass(frozen=True, slots=True)\nclass RecoveryWarning:\n    code: RecoveryCode\n    message: str\n    line_number: int\n\n\nclass SessionBusyError(RuntimeError):\n    """Structured failure raised when a Session already owns its writer lease."""\n\n    code = "session_busy"\n\n    def __init__(self, session_id: str, message: str | None = None) -> None:\n        self.session_id = session_id\n        super().__init__(\n            message or f"Session {session_id!r} already has an active writer"\n        )\n\n\nclass UnsupportedSchemaVersionError(ValueError):\n    def __init__(self, found_major: int) -> None:\n        self.artifact = "Session"\n        self.found_major = found_major\n        self.supported_major = SESSION_SCHEMA_VERSION.major\n        super().__init__(\n            f"Session schema major {found_major} is unsupported; "\n            f"this reader supports major {self.supported_major}. "\n            "Preserve the original and migrate it with migrate_session_file()."\n        )\n\n\ndef _validate_record_version(record: object, line_number: int) -> Mapping[str, object]:\n    if not isinstance(record, dict):\n        raise ValueError(f"Session record at line {line_number} must be an object")\n    if record.get("schema") != "agent_harness.session":\n        raise ValueError(f"invalid Session schema at line {line_number}")\n    raw_version = record.get("schema_version")\n    if not isinstance(raw_version, dict):\n        raise ValueError(f"invalid Session schema version at line {line_number}")\n    major = raw_version.get("major")\n    minor = raw_version.get("minor")\n    if (\n        isinstance(major, bool)\n        or not isinstance(major, int)\n        or isinstance(minor, bool)\n        or not isinstance(minor, int)\n        or major < 1\n        or minor < 0\n    ):\n        raise ValueError(f"invalid Session schema version at line {line_number}")\n    if major != SESSION_SCHEMA_VERSION.major:\n        raise UnsupportedSchemaVersionError(major)\n    return record\n\n\ndef _session_id(value: str) -> str:\n    if not _SESSION_ID.fullmatch(value):\n        raise ValueError("Session id must be a safe local identifier")\n    return value\n\n\ndef _version_record() -> dict[str, int]:\n    return {\n        "major": SESSION_SCHEMA_VERSION.major,\n        "minor": SESSION_SCHEMA_VERSION.minor,\n    }\n\n\ndef _encode_message(message: ConversationMessage) -> dict[str, object]:\n    if isinstance(message, AgentMessage):\n        content: list[dict[str, object]] = []\n        for block in message.content:\n            if isinstance(block, TextContent):\n                content.append({"type": "text", "text": block.text})\n            elif isinstance(block, ToolCallContent):\n                content.append(\n                    {\n                        "type": "tool_call",\n                        "id": block.id,\n                        "name": block.name,\n                        "arguments": block.arguments,\n                    }\n                )\n            else:\n                raise TypeError(f"unsupported Content Block: {type(block).__name__}")\n        return {\n            "kind": "agent_message",\n            "role": message.role.value,\n            "content": content,\n        }\n    if isinstance(message, ToolResultMessage):\n        result = message.result\n        truncation = (\n            None\n            if result.truncation is None\n            else {\n                "original_bytes": result.truncation.original_bytes,\n                "original_lines": result.truncation.original_lines,\n                "retained_start_byte": result.truncation.retained_start_byte,\n                "retained_end_byte": result.truncation.retained_end_byte,\n                "retained_start_line": result.truncation.retained_start_line,\n                "retained_end_line": result.truncation.retained_end_line,\n                "direction": result.truncation.direction.value,\n            }\n        )\n        complete_output = (\n            None\n            if result.complete_output is None\n            else {\n                "kind": result.complete_output.kind.value,\n                "reference": result.complete_output.reference,\n                "reason": result.complete_output.reason,\n            }\n        )\n        return {\n            "kind": "tool_result",\n            "tool_call_id": message.tool_call_id,\n            "tool_name": message.tool_name,\n            "result": {\n                "content": result.content,\n                "metadata": dict(result.metadata),\n                "terminate": result.terminate,\n                "is_error": result.is_error,\n                "error_code": (\n                    None if result.error_code is None else result.error_code.value\n                ),\n                "truncation": truncation,\n                "complete_output": complete_output,\n            },\n        }\n    raise TypeError(f"unsupported Session message: {type(message).__name__}")\n\n\ndef _decode_message(record: Mapping[str, object]) -> ConversationMessage:\n    if record.get("kind") == "agent_message":\n        role = Role(str(record["role"]))\n        raw_content = record.get("content")\n        if not isinstance(raw_content, list):\n            raise ValueError("Session AgentMessage content must be a list")\n        blocks: list[TextContent | ToolCallContent] = []\n        for raw_block in raw_content:\n            if not isinstance(raw_block, dict):\n                raise ValueError("Session Content Block must be an object")\n            if raw_block.get("type") == "text":\n                blocks.append(TextContent(str(raw_block["text"])))\n            elif raw_block.get("type") == "tool_call":\n                blocks.append(\n                    ToolCallContent(\n                        str(raw_block["id"]),\n                        str(raw_block["name"]),\n                        str(raw_block["arguments"]),\n                    )\n                )\n            else:\n                raise ValueError("unsupported Session Content Block type")\n        return AgentMessage(role, tuple(blocks))\n    if record.get("kind") == "tool_result":\n        raw_result = record.get("result")\n        if not isinstance(raw_result, dict):\n            raise ValueError("Session ToolResult must be an object")\n        raw_truncation = raw_result.get("truncation")\n        truncation = None\n        if isinstance(raw_truncation, dict):\n            truncation = TruncationNotice(\n                int(raw_truncation["original_bytes"]),\n                int(raw_truncation["original_lines"]),\n                int(raw_truncation["retained_start_byte"]),\n                int(raw_truncation["retained_end_byte"]),\n                int(raw_truncation["retained_start_line"]),\n                int(raw_truncation["retained_end_line"]),\n                TruncationDirection(str(raw_truncation["direction"])),\n            )\n        raw_complete = raw_result.get("complete_output")\n        complete_output = None\n        if isinstance(raw_complete, dict):\n            complete_output = CompleteOutputReference(\n                CompleteOutputKind(str(raw_complete["kind"])),\n                None\n                if raw_complete.get("reference") is None\n                else str(raw_complete["reference"]),\n                None\n                if raw_complete.get("reason") is None\n                else str(raw_complete["reason"]),\n            )\n        raw_metadata = raw_result.get("metadata", {})\n        if not isinstance(raw_metadata, dict):\n            raise ValueError("Session ToolResult metadata must be an object")\n        raw_error_code = raw_result.get("error_code")\n        result = ToolResult(\n            str(raw_result["content"]),\n            metadata=raw_metadata,\n            terminate=bool(raw_result.get("terminate", False)),\n            is_error=bool(raw_result.get("is_error", False)),\n            error_code=(\n                None\n                if raw_error_code is None\n                else ToolErrorCode(str(raw_error_code))\n            ),\n            truncation=truncation,\n            complete_output=complete_output,\n        )\n        return ToolResultMessage(\n            str(record["tool_call_id"]), str(record["tool_name"]), result\n        )\n    raise ValueError("unsupported Session message kind")\n\n\nclass SessionWriter(Protocol):\n    session_id: str\n\n    def __enter__(self) -> "SessionWriter": ...\n\n    def __exit__(self, *exc_info: object) -> None: ...\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> "SessionEntry": ...\n\n\nclass SessionStore(Protocol):\n    def create(self, session_id: str | None = None) -> "SessionState": ...\n\n    def read(self, session_id: str) -> "SessionState": ...\n\n    def writer(self, session_id: str) -> SessionWriter: ...\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionEntry:\n    entry_id: str\n    parent_id: str | None\n    messages: tuple[ConversationMessage, ...]\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionState:\n    session_id: str\n    entries: tuple[SessionEntry, ...] = ()\n    recovery_warning: RecoveryWarning | None = None\n\n    def __post_init__(self) -> None:\n        _session_id(self.session_id)\n        earlier: set[str] = set()\n        for entry in self.entries:\n            if not _SESSION_ID.fullmatch(entry.entry_id):\n                raise ValueError("Session entry id must be a safe local identifier")\n            if entry.entry_id in earlier:\n                raise ValueError(f"duplicate Session entry id: {entry.entry_id}")\n            if entry.parent_id is not None and entry.parent_id not in earlier:\n                raise ValueError(\n                    "Session entry parent must reference an earlier Session entry"\n                )\n            if not entry.messages:\n                raise ValueError("a settled Session entry requires messages")\n            earlier.add(entry.entry_id)\n\n    @property\n    def active_leaf_id(self) -> str | None:\n        return self.entries[-1].entry_id if self.entries else None\n\n    def history(self, leaf_id: str | None = None) -> tuple[ConversationMessage, ...]:\n        if not self.entries:\n            if leaf_id is not None:\n                raise KeyError(f"unknown Session entry: {leaf_id}")\n            return ()\n        by_id = {entry.entry_id: entry for entry in self.entries}\n        cursor = self.active_leaf_id if leaf_id is None else leaf_id\n        path: list[SessionEntry] = []\n        while cursor is not None:\n            try:\n                entry = by_id[cursor]\n            except KeyError:\n                raise KeyError(f"unknown Session entry: {cursor}") from None\n            path.append(entry)\n            cursor = entry.parent_id\n        return tuple(\n            message for entry in reversed(path) for message in entry.messages\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass MigrationResult:\n    source: Path\n    destination: Path\n    session_id: str\n    entries: int\n\n\nclass MemorySessionWriter:\n    def __init__(self, store: "MemorySessionStore", session_id: str) -> None:\n        self._store = store\n        self.session_id = session_id\n\n    def __enter__(self) -> "MemorySessionWriter":\n        return self\n\n    def __exit__(self, *exc_info: object) -> None:\n        return None\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("a settled Session entry requires messages")\n        entry = SessionEntry(self._store._id_factory(), parent, accepted)\n        self._store._sessions[self.session_id] = SessionState(\n            self.session_id, (*state.entries, entry)\n        )\n        return entry\n\n\nclass MemorySessionStore:\n    """In-process SessionStore adapter with the durable tree contract."""\n\n    def __init__(self, *, id_factory: IdFactory | None = None) -> None:\n        self._id_factory = id_factory or (lambda: uuid4().hex)\n        self._sessions: dict[str, SessionState] = {}\n\n    def create(self, session_id: str | None = None) -> SessionState:\n        accepted = _session_id(session_id or self._id_factory())\n        if accepted in self._sessions:\n            raise ValueError(f"Session already exists: {accepted}")\n        state = SessionState(accepted)\n        self._sessions[accepted] = state\n        return state\n\n    def read(self, session_id: str) -> SessionState:\n        try:\n            return self._sessions[session_id]\n        except KeyError:\n            raise KeyError(f"unknown Session: {session_id}") from None\n\n    def writer(self, session_id: str) -> MemorySessionWriter:\n        self.read(session_id)\n        return MemorySessionWriter(self, session_id)\n\n\nclass JSONLSessionWriter:\n    def __init__(self, store: "JSONLSessionStore", session_id: str) -> None:\n        self._store = store\n        self.session_id = session_id\n        self._lock_stream: BinaryIO | None = None\n\n    def __enter__(self) -> "JSONLSessionWriter":\n        if self._lock_stream is not None:\n            raise RuntimeError("Session writer lease is already active")\n        self._lock_stream = self._store._acquire_lock(self.session_id)\n        try:\n            state = self._store.read(self.session_id)\n            if state.recovery_warning is not None:\n                self._store._discard_uncommitted_tail(self.session_id)\n        except BaseException:\n            self._store._release_lock(self._lock_stream)\n            self._lock_stream = None\n            raise\n        return self\n\n    def __exit__(self, *exc_info: object) -> None:\n        if self._lock_stream is not None:\n            self._store._release_lock(self._lock_stream)\n            self._lock_stream = None\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if self._lock_stream is None:\n            raise RuntimeError("Session writer lease is not active")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("a settled Session entry requires messages")\n        entry = SessionEntry(self._store._id_factory(), parent, accepted)\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "settlement",\n            "session_id": self.session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "messages": [_encode_message(message) for message in accepted],\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        with self._store.path_for(self.session_id).open("a", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        return entry\n\n\nclass JSONLSessionStore:\n    """Transparent file-per-Session JSONL adapter."""\n\n    def __init__(self, root: str | Path, *, id_factory: IdFactory | None = None) -> None:\n        self.root = Path(root)\n        self.sessions_directory = self.root / "sessions"\n        self._id_factory = id_factory or (lambda: uuid4().hex)\n\n    def path_for(self, session_id: str) -> Path:\n        return self.sessions_directory / f"{_session_id(session_id)}.jsonl"\n\n    def _lock_path(self, session_id: str) -> Path:\n        return self.sessions_directory / ".locks" / f"{_session_id(session_id)}.lock"\n\n    def _discard_uncommitted_tail(self, session_id: str) -> None:\n        path = self.path_for(session_id)\n        content = path.read_bytes()\n        committed_end = content.rfind(b"\\n")\n        if committed_end < 0:\n            raise ValueError("Session file has no committed header")\n        with path.open("r+b") as stream:\n            stream.truncate(committed_end + 1)\n            stream.flush()\n            os.fsync(stream.fileno())\n\n    def _acquire_lock(self, session_id: str) -> BinaryIO:\n        path = self._lock_path(session_id)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        stream = path.open("a+b")\n        try:\n            if os.name == "nt":\n                import msvcrt\n\n                stream.seek(0, os.SEEK_END)\n                if stream.tell() == 0:\n                    stream.write(b"\\0")\n                    stream.flush()\n                stream.seek(0)\n                msvcrt.locking(  # type: ignore[attr-defined]\n                    stream.fileno(), msvcrt.LK_NBLCK, 1  # type: ignore[attr-defined]\n                )\n            else:\n                import fcntl\n\n                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except OSError:\n            stream.close()\n            raise SessionBusyError(session_id) from None\n        return stream\n\n    @staticmethod\n    def _release_lock(stream: BinaryIO) -> None:\n        try:\n            if os.name == "nt":\n                import msvcrt\n\n                stream.seek(0)\n                msvcrt.locking(  # type: ignore[attr-defined]\n                    stream.fileno(), msvcrt.LK_UNLCK, 1  # type: ignore[attr-defined]\n                )\n            else:\n                import fcntl\n\n                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)\n        finally:\n            stream.close()\n\n    def create(self, session_id: str | None = None) -> SessionState:\n        accepted = _session_id(session_id or self._id_factory())\n        path = self.path_for(accepted)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "session",\n            "session_id": accepted,\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        try:\n            with path.open("x", encoding="utf-8") as stream:\n                stream.write(encoded)\n                stream.flush()\n                os.fsync(stream.fileno())\n        except FileExistsError:\n            raise ValueError(f"Session already exists: {accepted}") from None\n        return SessionState(accepted)\n\n    def read(self, session_id: str) -> SessionState:\n        accepted = _session_id(session_id)\n        path = self.path_for(accepted)\n        try:\n            content = path.read_bytes()\n        except FileNotFoundError:\n            raise KeyError(f"unknown Session: {accepted}") from None\n        raw_lines = content.splitlines(keepends=True)\n        warning = None\n        if raw_lines and not raw_lines[-1].endswith(b"\\n"):\n            warning = RecoveryWarning(\n                RecoveryCode.INCOMPLETE_FINAL_RECORD,\n                "ignored an incomplete final Session record; restart the operation",\n                len(raw_lines),\n            )\n            raw_lines = raw_lines[:-1]\n        try:\n            lines = [line.decode("utf-8").rstrip("\\r\\n") for line in raw_lines]\n        except UnicodeDecodeError as error:\n            raise ValueError("Session JSONL must be UTF-8 text") from error\n        if not lines:\n            raise ValueError("Session file has no committed header")\n        entries: list[SessionEntry] = []\n        for line_number, line in enumerate(lines, 1):\n            try:\n                decoded = json.loads(line)\n            except json.JSONDecodeError as error:\n                raise ValueError(\n                    f"invalid Session JSONL record at line {line_number}"\n                ) from error\n            record = _validate_record_version(decoded, line_number)\n            if line_number == 1:\n                if record.get("record") != "session" or record.get("session_id") != accepted:\n                    raise ValueError("invalid Session header")\n                continue\n            if record.get("record") != "settlement" or record.get("session_id") != accepted:\n                raise ValueError(f"invalid Session record at line {line_number}")\n            raw_messages = record.get("messages")\n            if not isinstance(raw_messages, list) or not raw_messages:\n                raise ValueError("a settled Session entry requires messages")\n            entries.append(\n                SessionEntry(\n                    str(record["entry_id"]),\n                    None if record.get("parent_id") is None else str(record["parent_id"]),\n                    tuple(_decode_message(message) for message in raw_messages),\n                )\n            )\n        state = SessionState(accepted, tuple(entries), warning)\n        state.history()\n        return state\n\n    def writer(self, session_id: str) -> JSONLSessionWriter:\n        self.read(session_id)\n        return JSONLSessionWriter(self, session_id)\n\n\ndef migrate_session_file(\n    source: str | Path,\n    destination: str | Path,\n) -> MigrationResult:\n    """Write and validate a current Session file without changing its source."""\n\n    source_path = Path(source).resolve()\n    destination_path = Path(destination).resolve()\n    if source_path.parent.name != "sessions":\n        raise ValueError("source must be a file from a sessions directory")\n    session_id = _session_id(source_path.stem)\n    if destination_path.name != f"{session_id}.jsonl":\n        raise ValueError("migration destination must retain the Session filename")\n    if destination_path.exists():\n        raise FileExistsError(f"migration destination exists: {destination_path}")\n    state = JSONLSessionStore(source_path.parent.parent).read(session_id)\n    records: list[dict[str, object]] = [\n        {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "session",\n            "session_id": session_id,\n        }\n    ]\n    records.extend(\n        {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "settlement",\n            "session_id": session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "messages": [_encode_message(message) for message in entry.messages],\n        }\n        for entry in state.entries\n    )\n    encoded = "".join(\n        json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n        + "\\n"\n        for record in records\n    )\n    destination_path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(\n        prefix=".session-migration-", dir=destination_path.parent.parent\n    ) as temporary:\n        staging_root = Path(temporary)\n        staging = staging_root / "sessions" / f"{session_id}.jsonl"\n        staging.parent.mkdir(parents=True)\n        with staging.open("x", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        validated = JSONLSessionStore(staging_root).read(session_id)\n        if validated.history() != state.history():\n            raise ValueError("migrated Session failed history validation")\n        staging.replace(destination_path)\n    return MigrationResult(\n        source_path,\n        destination_path,\n        session_id,\n        len(state.entries),\n    )\n'

In [ ]:
RUNTIME_SOURCE = '"""Async Agent Runtime with structured Tool batches."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import AsyncIterator, Awaitable, Callable, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nimport json\nfrom typing import Protocol, TypeAlias, cast\n\nfrom jsonschema import (  # type: ignore[import-untyped]\n    Draft202012Validator,\n    ValidationError,\n)\n\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelRequest,\n    ModelSpec,\n    Role,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    Usage,\n    UsageUpdate,\n    to_model_messages,\n)\nfrom .tools import (\n    LocalToolExecutor,\n    PreparedToolCall,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    bound_tool_result,\n)\n\n\nclass EventType(str, Enum):\n    AGENT_START = "agent_start"\n    MODEL_ATTEMPT_START = "model_attempt_start"\n    MODEL_EVENT = "model_event"\n    MODEL_ATTEMPT_FAILED = "model_attempt_failed"\n    RETRY_SCHEDULED = "retry_scheduled"\n    TOOL_BATCH_START = "tool_batch_start"\n    TOOL_CALL_START = "tool_call_start"\n    TOOL_CALL_END = "tool_call_end"\n    TOOL_BATCH_END = "tool_batch_end"\n    RUN_CANCELLED = "run_cancelled"\n    MESSAGE_END = "message_end"\n    AGENT_END = "agent_end"\n\n\nclass TerminalStatus(str, Enum):\n    COMPLETED = "completed"\n    MODEL_ERROR = "model_error"\n    CANCELLED = "cancelled"\n    MAX_TURNS = "max_turns"\n    MAX_TOOL_CALLS = "max_tool_calls"\n    TIMEOUT = "timeout"\n    MAX_TOTAL_TOKENS = "max_total_tokens"\n\n\n@dataclass(frozen=True, slots=True)\nclass RunGuard:\n    max_turns: int | None = None\n    max_tool_calls: int | None = None\n    timeout_seconds: float | None = None\n    max_total_tokens: int | None = None\n\n    def __post_init__(self) -> None:\n        integer_limits = {\n            "max_turns": self.max_turns,\n            "max_tool_calls": self.max_tool_calls,\n            "max_total_tokens": self.max_total_tokens,\n        }\n        for name, value in integer_limits.items():\n            if value is not None and (\n                isinstance(value, bool) or not isinstance(value, int) or value <= 0\n            ):\n                raise ValueError(f"{name} must be a positive integer when supplied")\n        if self.timeout_seconds is not None and self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n\n    def reached(\n        self,\n        *,\n        turns: int,\n        tool_calls: int,\n        usage: Usage | None,\n        elapsed_seconds: float,\n    ) -> TerminalStatus | None:\n        if (\n            self.timeout_seconds is not None\n            and elapsed_seconds >= self.timeout_seconds\n        ):\n            return TerminalStatus.TIMEOUT\n        if self.max_turns is not None and turns >= self.max_turns:\n            return TerminalStatus.MAX_TURNS\n        if (\n            self.max_tool_calls is not None\n            and tool_calls >= self.max_tool_calls\n        ):\n            return TerminalStatus.MAX_TOOL_CALLS\n        if (\n            self.max_total_tokens is not None\n            and usage is not None\n            and usage.total_tokens >= self.max_total_tokens\n        ):\n            return TerminalStatus.MAX_TOTAL_TOKENS\n        return None\n\n\n@dataclass(frozen=True, slots=True)\nclass RuntimeEvent:\n    sequence: int\n    type: EventType\n    attempt: int | None = None\n    model_event: ModelEvent | None = None\n    error: ModelError | None = None\n    retry_delay_seconds: float | None = None\n    partial_text: str = ""\n    partial_usage: Usage | None = None\n    tool_call_id: str | None = None\n    tool_name: str | None = None\n    tool_result: ToolResult | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass AssistantOutcome:\n    message: AgentMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n    error: ModelError | None = None\n    attempts: int = 1\n    tool_results: tuple[ToolResult, ...] = ()\n    status: TerminalStatus = TerminalStatus.COMPLETED\n\n\n@dataclass(frozen=True, slots=True)\nclass RetryPolicy:\n    delays: tuple[float, ...] = (2.0, 4.0, 8.0)\n    max_retry_after_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        if any(delay < 0 for delay in self.delays):\n            raise ValueError("retry delays cannot be negative")\n        if self.max_retry_after_seconds < 0:\n            raise ValueError("max_retry_after_seconds cannot be negative")\n\n    def delay_for(\n        self,\n        error: ModelError,\n        failed_attempt: int,\n        *,\n        retry_after_seconds: float | None = None,\n    ) -> float | None:\n        retryable_codes = {\n            ModelErrorCode.RATE_LIMIT,\n            ModelErrorCode.TIMEOUT,\n            ModelErrorCode.CONNECTION,\n            ModelErrorCode.SERVER,\n        }\n        retryable_status = error.status_code in {408, 429} or (\n            error.status_code is not None and error.status_code >= 500\n        )\n        if (\n            error.code not in retryable_codes and not retryable_status\n        ) or failed_attempt > len(self.delays):\n            return None\n        if retry_after_seconds is not None:\n            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:\n                return None\n            return retry_after_seconds\n        return self.delays[failed_attempt - 1]\n\n\nSleeper: TypeAlias = Callable[[float], Awaitable[None]]\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nSettlementSink: TypeAlias = Callable[[Sequence[ConversationMessage]], None]\n_EVENTS_DONE = object()\n\n\nclass TurnInput(Protocol):\n    """Runtime-facing view of Steering Messages waiting at a turn boundary."""\n\n    def pending(self) -> bool: ...\n\n    def take(self) -> Sequence[AgentMessage]: ...\n\n\n@dataclass(slots=True)\nclass _CancellationState:\n    status: TerminalStatus = TerminalStatus.CANCELLED\n    requested: bool = False\n\n    def request(self, status: TerminalStatus) -> bool:\n        if self.requested:\n            return False\n        self.status = status\n        self.requested = True\n        return True\n\n\ndef _add_usage(left: Usage | None, right: Usage | None) -> Usage | None:\n    if left is None:\n        return right\n    if right is None:\n        return left\n    return Usage(\n        left.input_tokens + right.input_tokens,\n        left.output_tokens + right.output_tokens,\n        left.total_tokens + right.total_tokens,\n        left.estimated or right.estimated,\n    )\n\n\nclass AgentRunHandle:\n    """One accepted run\'s observations, cancellation, and eventual outcome."""\n\n    def __init__(\n        self,\n        task: asyncio.Task[AssistantOutcome],\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n    ) -> None:\n        self._task = task\n        self._events = events\n        self._cancellation = cancellation\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n    async def result(self) -> AssistantOutcome:\n        return await self._task\n\n    def cancel(self) -> None:\n        self._cancel_with(TerminalStatus.CANCELLED)\n\n    def _cancel_with(self, status: TerminalStatus) -> None:\n        if not self._task.done() and self._cancellation.request(status):\n            self._task.get_loop().call_soon(self._task.cancel)\n\n\nclass AgentRuntime:\n    """Advance typed conversation state through model and Tool turns."""\n\n    def __init__(\n        self,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        *,\n        tools: Sequence[Tool] = (),\n        tool_executor: ToolExecutor | None = None,\n        tool_output_budget: ToolOutputBudget | None = None,\n        retry_policy: RetryPolicy | None = None,\n        sleeper: Sleeper = asyncio.sleep,\n        run_guard: object | None = None,\n        history: Sequence[ConversationMessage] = (),\n    ) -> None:\n        if not isinstance(model, ModelSpec):\n            raise TypeError("model must be a ModelSpec")\n        if not callable(getattr(adapter, "stream", None)):\n            raise TypeError("adapter must implement ModelAdapter.stream")\n        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):\n            raise TypeError("retry_policy must be a RetryPolicy")\n        if not callable(sleeper):\n            raise TypeError("sleeper must be an async callable")\n        if tool_executor is not None and not callable(\n            getattr(tool_executor, "execute", None)\n        ):\n            raise TypeError("tool_executor must implement ToolExecutor.execute")\n        if tool_output_budget is not None and not isinstance(\n            tool_output_budget, ToolOutputBudget\n        ):\n            raise TypeError("tool_output_budget must be a ToolOutputBudget")\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._adapter = adapter\n        self._model = model\n        self._tools = registered\n        self._tool_executor = tool_executor or LocalToolExecutor()\n        self._tool_output_budget = tool_output_budget or ToolOutputBudget()\n        self._retry_policy = retry_policy or RetryPolicy()\n        self._sleeper = sleeper\n        self._run_guard = run_guard\n        accepted_history = tuple(history)\n        to_model_messages(accepted_history)\n        self._history: list[ConversationMessage] = list(accepted_history)\n        self._settlement_sink: SettlementSink | None = None\n\n    @property\n    def history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(self._history)\n\n    @property\n    def run_guard(self) -> object | None:\n        return self._run_guard\n\n    def restore_history(self, history: Sequence[ConversationMessage]) -> None:\n        """Seed a newly constructed Runtime from one settled Session branch."""\n\n        if self._history:\n            raise RuntimeError("Runtime history must be empty before restoration")\n        accepted = tuple(history)\n        to_model_messages(accepted)\n        self._history.extend(accepted)\n\n    def set_settlement_sink(self, sink: SettlementSink | None) -> None:\n        """Install the AgentSession-owned persistence barrier for the next Run."""\n\n        if sink is not None and not callable(sink):\n            raise TypeError("settlement sink must be callable")\n        self._settlement_sink = sink\n\n    def _append_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        self._history.extend(accepted)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def _mark_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def start(\n        self,\n        messages: Sequence[AgentMessage],\n        *,\n        turn_input: TurnInput | None = None,\n    ) -> AgentRunHandle:\n        accepted = tuple(messages)\n        to_model_messages((*self._history, *accepted))\n        self._append_settled(accepted)\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        cancellation = _CancellationState()\n        task = asyncio.get_running_loop().create_task(\n            self._execute(events, cancellation, turn_input)\n        )\n        handle = AgentRunHandle(task, events, cancellation)\n        if (\n            isinstance(self._run_guard, RunGuard)\n            and self._run_guard.timeout_seconds is not None\n        ):\n            timer = asyncio.get_running_loop().call_later(\n                self._run_guard.timeout_seconds,\n                handle._cancel_with,\n                TerminalStatus.TIMEOUT,\n            )\n            task.add_done_callback(lambda completed: timer.cancel())\n        return handle\n\n    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:\n        return await self.start(messages).result()\n\n    async def _execute(\n        self,\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        turn_input: TurnInput | None = None,\n    ) -> AssistantOutcome:\n        sequence = 0\n        total_attempts = 0\n        text_parts: list[str] = []\n        current_usage: Usage | None = None\n        run_usage: Usage | None = None\n        run_tool_results: list[ToolResult] = []\n        turns = 0\n        tool_calls = 0\n        started_at = asyncio.get_running_loop().time()\n\n        def guard_status() -> TerminalStatus | None:\n            if not isinstance(self._run_guard, RunGuard):\n                return None\n            return self._run_guard.reached(\n                turns=turns,\n                tool_calls=tool_calls,\n                usage=run_usage,\n                elapsed_seconds=asyncio.get_running_loop().time() - started_at,\n            )\n\n        async def emit(\n            type_: EventType,\n            *,\n            attempt: int | None = None,\n            model_event: ModelEvent | None = None,\n            error: ModelError | None = None,\n            retry_delay_seconds: float | None = None,\n            partial_text: str = "",\n            partial_usage: Usage | None = None,\n            tool_call_id: str | None = None,\n            tool_name: str | None = None,\n            tool_result: ToolResult | None = None,\n        ) -> None:\n            nonlocal sequence\n            sequence += 1\n            await events.put(\n                RuntimeEvent(\n                    sequence=sequence,\n                    type=type_,\n                    attempt=attempt,\n                    model_event=model_event,\n                    error=error,\n                    retry_delay_seconds=retry_delay_seconds,\n                    partial_text=partial_text,\n                    partial_usage=partial_usage,\n                    tool_call_id=tool_call_id,\n                    tool_name=tool_name,\n                    tool_result=tool_result,\n                )\n            )\n\n        async def finish(outcome: AssistantOutcome) -> AssistantOutcome:\n            self._append_settled((outcome.message,))\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n\n        try:\n            await emit(EventType.AGENT_START)\n            while True:\n                turn_attempt = 0\n                while True:\n                    turn_attempt += 1\n                    total_attempts += 1\n                    attempt = total_attempts\n                    request = ModelRequest(\n                        to_model_messages(self._history),\n                        self._model,\n                        tuple(tool.definition() for tool in self._tools.values()),\n                    )\n                    await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)\n                    text_parts = []\n                    tool_drafts: dict[int, dict[str, str]] = {}\n                    current_usage = None\n                    end: ModelEnd | None = None\n                    schema_error: ModelError | None = None\n                    try:\n                        async for event in self._adapter.stream(request):\n                            await emit(\n                                EventType.MODEL_EVENT,\n                                attempt=attempt,\n                                model_event=event,\n                            )\n                            if end is not None:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted data after ModelEnd",\n                                    False,\n                                )\n                                break\n                            if isinstance(event, TextDelta):\n                                text_parts.append(event.text)\n                            elif isinstance(event, ToolCallDelta):\n                                if event.index < 0:\n                                    schema_error = ModelError(\n                                        ModelErrorCode.SCHEMA,\n                                        "model stream emitted an invalid Tool Call index",\n                                        False,\n                                    )\n                                    break\n                                draft = tool_drafts.setdefault(\n                                    event.index,\n                                    {"id": "", "name": "", "arguments": ""},\n                                )\n                                draft["id"] += event.id\n                                draft["name"] += event.name\n                                draft["arguments"] += event.arguments_delta\n                            elif isinstance(event, UsageUpdate):\n                                current_usage = event.usage\n                            elif isinstance(event, ModelEnd):\n                                end = event\n                            else:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted an unsupported event",\n                                    False,\n                                )\n                                break\n                    except ModelAdapterError as failure:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=failure.error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        delay = self._retry_policy.delay_for(\n                            failure.error,\n                            turn_attempt,\n                            retry_after_seconds=failure.error.retry_after_seconds,\n                        )\n                        if delay is not None:\n                            await emit(\n                                EventType.RETRY_SCHEDULED,\n                                attempt=attempt,\n                                error=failure.error,\n                                retry_delay_seconds=delay,\n                                partial_text=partial_text,\n                                partial_usage=current_usage,\n                            )\n                            text_parts = []\n                            current_usage = None\n                            await self._sleeper(delay)\n                            continue\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                failure.error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n\n                    error = schema_error\n                    if error is None and end is None:\n                        error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "model stream violated the provider-neutral event contract",\n                            False,\n                        )\n                    blocks: list[ContentBlock] = []\n                    if error is None:\n                        try:\n                            if text_parts:\n                                blocks.append(TextContent("".join(text_parts)))\n                            for index in sorted(tool_drafts):\n                                blocks.append(ToolCallContent(**tool_drafts[index]))\n                        except (TypeError, ValueError):\n                            error = ModelError(\n                                ModelErrorCode.SCHEMA,\n                                "model stream emitted an incomplete Tool Call",\n                                False,\n                            )\n                    if error is not None:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    assert end is not None\n                    break\n\n                run_usage = _add_usage(run_usage, current_usage)\n                turns += 1\n                assistant = AgentMessage(Role.ASSISTANT, tuple(blocks))\n                calls = tuple(\n                    block\n                    for block in assistant.content\n                    if isinstance(block, ToolCallContent)\n                )\n                if calls:\n                    self._history.append(assistant)\n                else:\n                    self._append_settled((assistant,))\n                await emit(EventType.MESSAGE_END, attempt=total_attempts)\n                if not calls:\n                    if turn_input is not None and turn_input.pending():\n                        reached = guard_status()\n                        if reached is not None:\n                            await emit(EventType.AGENT_END, attempt=total_attempts)\n                            return AssistantOutcome(\n                                assistant,\n                                StopReason.ABORTED,\n                                run_usage,\n                                attempts=total_attempts,\n                                tool_results=tuple(run_tool_results),\n                                status=reached,\n                            )\n                        steering = tuple(turn_input.take())\n                        to_model_messages(steering)\n                        self._append_settled(steering)\n                        continue\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n\n                await emit(EventType.TOOL_BATCH_START, attempt=total_attempts)\n                prepared: dict[int, PreparedToolCall] = {}\n                results: dict[int, ToolResult] = {}\n                for index, call in enumerate(calls):\n                    tool = self._tools.get(call.name)\n                    if tool is None:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.UNKNOWN_TOOL, call.name\n                        )\n                        continue\n                    try:\n                        parsed = json.loads(call.arguments)\n                    except (json.JSONDecodeError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_JSON, call.name\n                        )\n                        continue\n                    try:\n                        Draft202012Validator(tool.input_schema).validate(parsed)\n                    except ValidationError:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    if not isinstance(parsed, dict):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    prepared[index] = PreparedToolCall(call, tool, parsed)\n\n                async def execute_one(index: int, call: PreparedToolCall) -> None:\n                    await emit(\n                        EventType.TOOL_CALL_START,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                    )\n                    try:\n                        result = await self._tool_executor.execute(call)\n                        if not isinstance(result, ToolResult):\n                            raise TypeError("ToolExecutor returned an invalid result")\n                    except Exception:\n                        result = ToolResult.error(\n                            ToolErrorCode.EXECUTION_FAILED, call.call.name\n                        )\n                    result = bound_tool_result(\n                        result,\n                        self._tool_output_budget,\n                        call.tool.output_direction,\n                    )\n                    results[index] = result\n                    await emit(\n                        EventType.TOOL_CALL_END,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                        tool_result=result,\n                    )\n\n                try:\n                    if any(call.tool.sequential for call in prepared.values()):\n                        for index, prepared_call in prepared.items():\n                            await execute_one(index, prepared_call)\n                    else:\n                        tasks = {\n                            index: asyncio.create_task(\n                                execute_one(index, prepared_call)\n                            )\n                            for index, prepared_call in prepared.items()\n                        }\n                        try:\n                            await asyncio.gather(*tasks.values())\n                        except asyncio.CancelledError:\n                            for task in tasks.values():\n                                if not task.done():\n                                    task.cancel()\n                            await asyncio.gather(\n                                *tasks.values(), return_exceptions=True\n                            )\n                            raise\n                except asyncio.CancelledError:\n                    for index, prepared_call in prepared.items():\n                        if index not in results:\n                            cancelled_result = ToolResult.error(\n                                ToolErrorCode.CANCELLED,\n                                prepared_call.call.name,\n                            )\n                            results[index] = cancelled_result\n                            await emit(\n                                EventType.TOOL_CALL_END,\n                                attempt=total_attempts,\n                                tool_call_id=prepared_call.call.id,\n                                tool_name=prepared_call.call.name,\n                                tool_result=cancelled_result,\n                            )\n                    for index, call in enumerate(calls):\n                        result = results[index]\n                        run_tool_results.append(result)\n                        self._history.append(\n                            ToolResultMessage(call.id, call.name, result)\n                        )\n                    self._mark_settled(self._history[-(len(calls) + 1) :])\n                    await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                    raise\n\n                batch_results: list[ToolResult] = []\n                for index, call in enumerate(calls):\n                    result = results[index]\n                    batch_results.append(result)\n                    run_tool_results.append(result)\n                    self._history.append(\n                        ToolResultMessage(call.id, call.name, result)\n                    )\n                self._mark_settled(self._history[-(len(calls) + 1) :])\n                tool_calls += len(calls)\n                await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                if batch_results and all(result.terminate for result in batch_results):\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n                reached = guard_status()\n                if reached is not None:\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        StopReason.ABORTED,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                        status=reached,\n                    )\n                if turn_input is not None and turn_input.pending():\n                    steering = tuple(turn_input.take())\n                    to_model_messages(steering)\n                    self._append_settled(steering)\n        except asyncio.CancelledError:\n            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))\n            outcome = AssistantOutcome(\n                message,\n                StopReason.ABORTED,\n                _add_usage(run_usage, current_usage),\n                attempts=max(total_attempts, 1),\n                tool_results=tuple(run_tool_results),\n                status=cancellation.status,\n            )\n            self._append_settled((message,))\n            await emit(\n                EventType.RUN_CANCELLED,\n                attempt=max(total_attempts, 1),\n                partial_text="".join(text_parts),\n                partial_usage=current_usage,\n            )\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n        finally:\n            await events.put(_EVENTS_DONE)\n'

In [ ]:
SESSION_SOURCE = '"""Application-facing control for one in-memory agent conversation."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections import deque\nfrom dataclasses import dataclass\nfrom collections.abc import AsyncIterator, Sequence\nfrom enum import Enum\nfrom typing import cast\n\nfrom .model import AgentMessage, Role, StopReason\nfrom .persistence import SessionBusyError, SessionStore, SessionWriter\nfrom .runtime import AgentRunHandle, AgentRuntime, AssistantOutcome, RuntimeEvent\n\n\n_SESSION_EVENTS_DONE = object()\n\n\nclass InputKind(str, Enum):\n    STEERING = "steering"\n    FOLLOW_UP = "follow_up"\n\n\n@dataclass(frozen=True, slots=True)\nclass PendingInput:\n    kind: InputKind\n    message: AgentMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionRunResult:\n    outcome: AssistantOutcome\n    outcomes: tuple[AssistantOutcome, ...] = ()\n    pending_inputs: tuple[PendingInput, ...] = ()\n\n\nclass SessionRunHandle:\n    def __init__(\n        self,\n        task: asyncio.Task[SessionRunResult],\n        session: "AgentSession",\n        events: asyncio.Queue[RuntimeEvent | object],\n    ) -> None:\n        self._task = task\n        self._session = session\n        self._events = events\n\n    async def result(self) -> SessionRunResult:\n        return await self._task\n\n    def cancel(self) -> None:\n        if not self._task.done():\n            self._session.cancel()\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _SESSION_EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n\nclass _SteeringQueue:\n    def __init__(self) -> None:\n        self._messages: deque[PendingInput] = deque()\n\n    def append(self, message: AgentMessage) -> None:\n        self._messages.append(PendingInput(InputKind.STEERING, message))\n\n    def pending(self) -> bool:\n        return bool(self._messages)\n\n    def take(self) -> Sequence[AgentMessage]:\n        return (self._messages.popleft().message,)\n\n    def drain(self) -> tuple[PendingInput, ...]:\n        drained = tuple(self._messages)\n        self._messages.clear()\n        return drained\n\n\nclass AgentSession:\n    """Coordinate one active Run with optional durable Session state."""\n\n    def __init__(\n        self,\n        runtime: AgentRuntime,\n        *,\n        store: SessionStore | None = None,\n        session_id: str | None = None,\n        parent_entry_id: str | None = None,\n    ) -> None:\n        if not isinstance(runtime, AgentRuntime):\n            raise TypeError("runtime must be an AgentRuntime")\n        if (store is None) != (session_id is None):\n            raise ValueError("durable Sessions require both store and session_id")\n        self._runtime = runtime\n        self._store = store\n        self._session_id = session_id\n        self._parent_entry_id = parent_entry_id\n        self._busy = False\n        self._active: AgentRunHandle | None = None\n        self._cancel_requested = False\n        self._steering = _SteeringQueue()\n        self._follow_ups: deque[PendingInput] = deque()\n\n    @property\n    def busy(self) -> bool:\n        return self._busy\n\n    @property\n    def session_id(self) -> str | None:\n        return self._session_id\n\n    def start(self, prompt: str | AgentMessage) -> SessionRunHandle:\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Session already has an active Run",\n            )\n        message = (\n            AgentMessage.text(Role.USER, prompt) if isinstance(prompt, str) else prompt\n        )\n        if not isinstance(message, AgentMessage) or message.role is not Role.USER:\n            raise TypeError("prompt must be text or a user AgentMessage")\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id)\n            writer = writer.__enter__()\n        self._busy = True\n        self._cancel_requested = False\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        try:\n            task = asyncio.get_running_loop().create_task(\n                self._drive(message, events, writer)\n            )\n        except BaseException:\n            self._busy = False\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            raise\n        return SessionRunHandle(task, self, events)\n\n    async def run(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.start(prompt).result()\n\n    async def prompt(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.run(prompt)\n\n    def steer(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Steering requires an active Run")\n        self._steering.append(self._user_message(message))\n\n    def follow_up(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Follow-up requires an active Run")\n        self._follow_ups.append(\n            PendingInput(InputKind.FOLLOW_UP, self._user_message(message))\n        )\n\n    def cancel(self) -> None:\n        self._cancel_requested = True\n        if self._active is not None:\n            self._active.cancel()\n\n    async def _drive(\n        self,\n        message: AgentMessage,\n        events: asyncio.Queue[RuntimeEvent | object],\n        writer: SessionWriter | None,\n    ) -> SessionRunResult:\n        outcomes: list[AssistantOutcome] = []\n\n        if writer is not None:\n            def settle(messages) -> None:\n                entry = writer.append(messages, parent_id=self._parent_entry_id)\n                self._parent_entry_id = entry.entry_id\n\n            self._runtime.set_settlement_sink(settle)\n\n        async def forward(handle: AgentRunHandle) -> None:\n            async for event in handle.events():\n                await events.put(event)\n\n        try:\n            next_message = message\n            while True:\n                self._active = self._runtime.start(\n                    [next_message], turn_input=self._steering\n                )\n                if self._cancel_requested:\n                    self._active.cancel()\n                forwarding = asyncio.create_task(forward(self._active))\n                outcome = await self._active.result()\n                await forwarding\n                outcomes.append(outcome)\n                if outcome.stop_reason in {StopReason.ABORTED, StopReason.ERROR}:\n                    break\n                if not self._follow_ups:\n                    break\n                next_message = self._follow_ups.popleft().message\n            pending = self._steering.drain() + tuple(self._follow_ups)\n            self._follow_ups.clear()\n            return SessionRunResult(outcome, tuple(outcomes), pending)\n        finally:\n            self._runtime.set_settlement_sink(None)\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._active = None\n            self._cancel_requested = False\n            self._busy = False\n            await events.put(_SESSION_EVENTS_DONE)\n\n    @staticmethod\n    def _user_message(message: str | AgentMessage) -> AgentMessage:\n        accepted = (\n            AgentMessage.text(Role.USER, message)\n            if isinstance(message, str)\n            else message\n        )\n        if not isinstance(accepted, AgentMessage) or accepted.role is not Role.USER:\n            raise TypeError("Session input must be text or a user AgentMessage")\n        return accepted\n\n\ndef create_agent_session(\n    runtime: AgentRuntime,\n    *,\n    store: SessionStore | None = None,\n    session_id: str | None = None,\n    fork_from: str | None = None,\n    no_save: bool = False,\n) -> AgentSession:\n    """Create a new durable Session, explicitly continue/fork one, or opt out."""\n\n    if no_save:\n        if store is not None or session_id is not None or fork_from is not None:\n            raise ValueError("no_save cannot be combined with persistence or continuation")\n        return AgentSession(runtime)\n    if store is None:\n        if session_id is not None or fork_from is not None:\n            raise ValueError("continuation requires a SessionStore")\n        return AgentSession(runtime)\n    if runtime.history:\n        raise ValueError("a durable AgentSession requires a fresh AgentRuntime")\n    if session_id is None:\n        if fork_from is not None:\n            raise ValueError("fork_from requires an existing session_id")\n        state = store.create()\n        leaf = None\n    else:\n        state = store.read(session_id)\n        leaf = fork_from if fork_from is not None else state.active_leaf_id\n        runtime.restore_history(state.history(leaf))\n    return AgentSession(\n        runtime,\n        store=store,\n        session_id=state.session_id,\n        parent_entry_id=leaf,\n    )\n'

In [ ]:
INIT_SOURCE = 'from .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelMessage,\n    ModelProtocolError,\n    ModelRequest,\n    ModelResult,\n    ModelSpec,\n    ModelToolResultMessage,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    UnsupportedContentError,\n    Usage,\n    UsageUpdate,\n    complete,\n    to_model_messages,\n)\nfrom .persistence import (\n    ConversationMessage,\n    JSONLSessionStore,\n    JSONLSessionWriter,\n    MemorySessionStore,\n    MemorySessionWriter,\n    MigrationResult,\n    RecoveryCode,\n    RecoveryWarning,\n    SESSION_SCHEMA_VERSION,\n    SchemaVersion,\n    SessionBusyError,\n    SessionEntry,\n    SessionStore,\n    SessionState,\n    SessionWriter,\n    UnsupportedSchemaVersionError,\n    migrate_session_file,\n)\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    EventType,\n    RetryPolicy,\n    RunGuard,\n    RuntimeEvent,\n    Sleeper,\n    TerminalStatus,\n    TurnInput,\n)\nfrom .session import (\n    AgentSession,\n    InputKind,\n    PendingInput,\n    SessionRunHandle,\n    SessionRunResult,\n    create_agent_session,\n)\nfrom .tools import (\n    AsyncioProcessOperations,\n    CompleteOutputKind,\n    CompleteOutputReference,\n    LocalToolExecutor,\n    PreparedToolCall,\n    ProcessResult,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n__all__ = [name for name in globals() if not name.startswith("_")]\n'

In [ ]:
import asyncio
import importlib
from tempfile import TemporaryDirectory

MODEL_SOURCE = (CHAPTER_4 / 'src' / 'agent_harness' / 'model.py').read_text(encoding='utf-8')
TOOLS_SOURCE = (CHAPTER_4 / 'src' / 'agent_harness' / 'tools.py').read_text(encoding='utf-8')
with TemporaryDirectory(prefix='chapter-05-minimal-') as temporary:
    package = Path(temporary) / 'agent_harness'
    package.mkdir()
    for name, text in {
        'model.py': MODEL_SOURCE, 'tools.py': TOOLS_SOURCE,
        'runtime.py': RUNTIME_SOURCE, 'persistence.py': PERSISTENCE_SOURCE,
        'session.py': SESSION_SOURCE, '__init__.py': INIT_SOURCE,
    }.items():
        (package / name).write_text(text, encoding='utf-8')
    sys.path.insert(0, temporary)
    chapter5 = importlib.import_module('agent_harness')

    async def minimal_run():
        adapter = chapter5.ScriptedModelAdapter([
            chapter5.TextDelta('durable answer'),
            chapter5.ModelEnd(chapter5.StopReason.COMPLETE),
        ])
        store = chapter5.MemorySessionStore()
        session = chapter5.create_agent_session(
            chapter5.AgentRuntime(adapter, chapter5.ModelSpec('scripted/ch05')),
            store=store,
        )
        result = await session.run('persist me')
        state = store.read(session.session_id)
        return result, state

    minimal_result, minimal_state = await minimal_run()
    assert minimal_result.outcome.message == chapter5.AgentMessage.text(
        chapter5.Role.ASSISTANT, 'durable answer'
    )
    assert len(minimal_state.entries) == 2
    sys.path.remove(temporary)
    for module_name in tuple(sys.modules):
        if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
            del sys.modules[module_name]

## Staged Construction

The tracer bullet first made one settled batch recoverable through the Memory adapter. The same interface then drove a transparent JSONL adapter. Successive red–green slices added complete model–Tool reconstruction, incomplete-tail recovery, operating-system writer leases, schema-major checks, non-destructive migration, explicit continuation, historical forks, and the ephemeral privacy path.

In [ ]:
DURABLE_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nimport pytest\n\nfrom agent_harness import (\n    AgentMessage,\n    AgentRuntime,\n    JSONLSessionStore,\n    MemorySessionStore,\n    ModelEnd,\n    ModelSpec,\n    MigrationResult,\n    RecoveryCode,\n    Role,\n    ScriptedModelAdapter,\n    SessionBusyError,\n    StopReason,\n    TextDelta,\n    Tool,\n    ToolCallDelta,\n    ToolResult,\n    UnsupportedSchemaVersionError,\n    create_agent_session,\n    migrate_session_file,\n)\n\n\ndef test_memory_store_recovers_a_settled_message_batch() -> None:\n    store = MemorySessionStore()\n    created = store.create("session-1")\n\n    with store.writer(created.session_id) as writer:\n        entry = writer.append(\n            (\n                AgentMessage.text(Role.USER, "hello"),\n                AgentMessage.text(Role.ASSISTANT, "hi"),\n            )\n        )\n\n    recovered = store.read("session-1")\n\n    assert recovered.active_leaf_id == entry.entry_id\n    assert recovered.history() == (\n        AgentMessage.text(Role.USER, "hello"),\n        AgentMessage.text(Role.ASSISTANT, "hi"),\n    )\n\n\ndef test_jsonl_store_recovers_through_the_same_public_interface(tmp_path) -> None:\n    store = JSONLSessionStore(tmp_path)\n    state = store.create("session-on-disk")\n\n    with store.writer(state.session_id) as writer:\n        writer.append(\n            (\n                AgentMessage.text(Role.USER, "persist this"),\n                AgentMessage.text(Role.ASSISTANT, "persisted"),\n            )\n        )\n\n    reopened = JSONLSessionStore(tmp_path).read(state.session_id)\n\n    assert reopened.history() == (\n        AgentMessage.text(Role.USER, "persist this"),\n        AgentMessage.text(Role.ASSISTANT, "persisted"),\n    )\n    assert (tmp_path / "sessions" / "session-on-disk.jsonl").is_file()\n\n\ndef test_model_and_tool_history_survives_reconstruction_and_continues(tmp_path) -> None:\n    async def scenario() -> None:\n        async def lookup(arguments: dict[str, object]) -> ToolResult:\n            return ToolResult("durable tool result", metadata={"source": "fixture"})\n\n        first_adapter = ScriptedModelAdapter(\n            [\n                [\n                    ToolCallDelta(0, "call-1", "lookup", \'{"key":"value"}\'),\n                    ModelEnd(StopReason.TOOL_USE),\n                ],\n                [TextDelta("first answer"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        store = JSONLSessionStore(tmp_path)\n        first = create_agent_session(\n            AgentRuntime(\n                first_adapter,\n                ModelSpec("scripted/durable"),\n                tools=[\n                    Tool(\n                        "lookup",\n                        "Look up a fixture value",\n                        {\n                            "type": "object",\n                            "properties": {"key": {"type": "string"}},\n                            "required": ["key"],\n                            "additionalProperties": False,\n                        },\n                        lookup,\n                    )\n                ],\n            ),\n            store=store,\n        )\n        first_result = await first.run("first question")\n        assert first_result.outcome.message == AgentMessage.text(\n            Role.ASSISTANT, "first answer"\n        )\n\n        second_adapter = ScriptedModelAdapter(\n            [TextDelta("continued answer"), ModelEnd(StopReason.COMPLETE)]\n        )\n        continued = create_agent_session(\n            AgentRuntime(second_adapter, ModelSpec("scripted/durable")),\n            store=JSONLSessionStore(tmp_path),\n            session_id=first.session_id,\n        )\n        second_result = await continued.run("follow up")\n\n        assert second_result.outcome.message == AgentMessage.text(\n            Role.ASSISTANT, "continued answer"\n        )\n        request = second_adapter.received_requests[0]\n        assert len(request.messages) == 5\n        assert request.messages[-1] == AgentMessage.text(\n            Role.USER, "follow up"\n        ).to_model()\n        assert request.messages[2].role is Role.TOOL\n\n    asyncio.run(scenario())\n\n\ndef test_incomplete_final_jsonl_record_is_reported_and_not_replayed(tmp_path) -> None:\n    store = JSONLSessionStore(tmp_path)\n    state = store.create("interrupted")\n    with store.writer(state.session_id) as writer:\n        writer.append((AgentMessage.text(Role.USER, "settled"),))\n    path = store.path_for(state.session_id)\n    with path.open("ab") as stream:\n        stream.write(b\'{"record":"settlement","entry_id":"interrupted"\')\n\n    recovered = JSONLSessionStore(tmp_path).read(state.session_id)\n\n    assert recovered.history() == (AgentMessage.text(Role.USER, "settled"),)\n    assert recovered.recovery_warning is not None\n    assert recovered.recovery_warning.code is RecoveryCode.INCOMPLETE_FINAL_RECORD\n\n\ndef test_continuation_discards_an_uncommitted_tail_before_appending(tmp_path) -> None:\n    async def scenario() -> None:\n        store = JSONLSessionStore(tmp_path)\n        state = store.create("recover-and-continue")\n        with store.writer(state.session_id) as writer:\n            writer.append((AgentMessage.text(Role.USER, "settled before crash"),))\n        with store.path_for(state.session_id).open("ab") as stream:\n            stream.write(b\'{"record":"settlement","messages":[\')\n\n        continued = create_agent_session(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("settled after restart"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/recovery"),\n            ),\n            store=JSONLSessionStore(tmp_path),\n            session_id=state.session_id,\n        )\n        await continued.run("restart operation")\n\n        recovered = JSONLSessionStore(tmp_path).read(state.session_id)\n        assert recovered.recovery_warning is None\n        assert recovered.history() == (\n            AgentMessage.text(Role.USER, "settled before crash"),\n            AgentMessage.text(Role.USER, "restart operation"),\n            AgentMessage.text(Role.ASSISTANT, "settled after restart"),\n        )\n\n    asyncio.run(scenario())\n\n\ndef test_durable_session_has_one_writer_while_reads_and_other_sessions_continue(\n    tmp_path,\n) -> None:\n    first_store = JSONLSessionStore(tmp_path)\n    first_store.create("shared")\n    first_store.create("independent")\n    competing_store = JSONLSessionStore(tmp_path)\n\n    with first_store.writer("shared"):\n        with pytest.raises(SessionBusyError) as caught:\n            with competing_store.writer("shared"):\n                pass\n        assert caught.value.code == "session_busy"\n        assert caught.value.session_id == "shared"\n        assert competing_store.read("shared").session_id == "shared"\n        with competing_store.writer("independent") as writer:\n            writer.append((AgentMessage.text(Role.USER, "unblocked"),))\n\n\ndef test_session_writer_lease_contends_across_processes(tmp_path) -> None:\n    store = JSONLSessionStore(tmp_path)\n    store.create("cross-process")\n    ready = tmp_path / "child-ready"\n    release = tmp_path / "child-release"\n    script = """\nimport sys\nimport time\nfrom pathlib import Path\nfrom agent_harness import JSONLSessionStore\n\nroot, ready, release = map(Path, sys.argv[1:])\nwith JSONLSessionStore(root).writer("cross-process"):\n    ready.write_text("ready", encoding="utf-8")\n    while not release.exists():\n        time.sleep(0.01)\n"""\n    environment = dict(os.environ)\n    source_root = str(Path(__file__).resolve().parents[1] / "src")\n    environment["PYTHONPATH"] = os.pathsep.join(\n        item for item in (source_root, environment.get("PYTHONPATH", "")) if item\n    )\n    child = subprocess.Popen(\n        [sys.executable, "-c", script, str(tmp_path), str(ready), str(release)],\n        env=environment,\n    )\n    try:\n        deadline = time.monotonic() + 5\n        while not ready.exists() and child.poll() is None and time.monotonic() < deadline:\n            time.sleep(0.01)\n        assert ready.exists(), "child process did not acquire the Session Lock"\n        with pytest.raises(SessionBusyError):\n            with store.writer("cross-process"):\n                pass\n    finally:\n        release.write_text("release", encoding="utf-8")\n        child.wait(timeout=5)\n    assert child.returncode == 0\n\n\ndef test_session_schema_accepts_optional_fields_and_rejects_unknown_major(\n    tmp_path,\n) -> None:\n    store = JSONLSessionStore(tmp_path)\n    state = store.create("versioned")\n    with store.writer(state.session_id) as writer:\n        writer.append((AgentMessage.text(Role.USER, "compatible"),))\n    path = store.path_for(state.session_id)\n    records = [json.loads(line) for line in path.read_text().splitlines()]\n    records[0]["future_header_field"] = {"safe": True}\n    records[1]["future_entry_field"] = [1, 2, 3]\n    path.write_text("\\n".join(json.dumps(record) for record in records) + "\\n")\n\n    assert JSONLSessionStore(tmp_path).read(state.session_id).history() == (\n        AgentMessage.text(Role.USER, "compatible"),\n    )\n\n    records[0]["schema_version"] = {"major": 2, "minor": 0}\n    path.write_text("\\n".join(json.dumps(record) for record in records) + "\\n")\n    with pytest.raises(\n        UnsupportedSchemaVersionError,\n        match=r"Session schema major 2.*migrate",\n    ):\n        JSONLSessionStore(tmp_path).read(state.session_id)\n\n\ndef test_complete_jsonl_record_cannot_reference_an_unknown_parent(tmp_path) -> None:\n    store = JSONLSessionStore(tmp_path)\n    state = store.create("invalid-tree")\n    with store.writer(state.session_id) as writer:\n        writer.append((AgentMessage.text(Role.USER, "root"),))\n    path = store.path_for(state.session_id)\n    records = [json.loads(line) for line in path.read_text().splitlines()]\n    records[1]["parent_id"] = "missing-parent"\n    path.write_text("\\n".join(json.dumps(record) for record in records) + "\\n")\n\n    with pytest.raises(ValueError, match="parent.*earlier Session entry"):\n        JSONLSessionStore(tmp_path).read(state.session_id)\n\n\ndef test_migration_validates_a_new_file_and_preserves_the_original(tmp_path) -> None:\n    source_store = JSONLSessionStore(tmp_path / "source")\n    state = source_store.create("migration-fixture")\n    with source_store.writer(state.session_id) as writer:\n        writer.append((AgentMessage.text(Role.USER, "keep the original"),))\n    source = source_store.path_for(state.session_id)\n    original = source.read_bytes()\n    destination = (\n        tmp_path / "migrated" / "sessions" / "migration-fixture.jsonl"\n    )\n\n    result = migrate_session_file(source, destination)\n\n    assert result == MigrationResult(\n        source=source.resolve(),\n        destination=destination.resolve(),\n        session_id="migration-fixture",\n        entries=1,\n    )\n    assert source.read_bytes() == original\n    assert destination.read_bytes() != b""\n    assert JSONLSessionStore(tmp_path / "migrated").read(state.session_id).history() == (\n        AgentMessage.text(Role.USER, "keep the original"),\n    )\n\n\ndef test_both_store_adapters_preserve_history_when_a_historical_entry_forks(\n    tmp_path,\n) -> None:\n    for store in (MemorySessionStore(), JSONLSessionStore(tmp_path)):\n        state = store.create(f"tree-{type(store).__name__}")\n        with store.writer(state.session_id) as writer:\n            root = writer.append((AgentMessage.text(Role.USER, "root"),))\n            main = writer.append((AgentMessage.text(Role.ASSISTANT, "main"),))\n            branch = writer.append(\n                (AgentMessage.text(Role.ASSISTANT, "branch"),),\n                parent_id=root.entry_id,\n            )\n\n        recovered = store.read(state.session_id)\n\n        assert recovered.history(main.entry_id) == (\n            AgentMessage.text(Role.USER, "root"),\n            AgentMessage.text(Role.ASSISTANT, "main"),\n        )\n        assert recovered.history() == (\n            AgentMessage.text(Role.USER, "root"),\n            AgentMessage.text(Role.ASSISTANT, "branch"),\n        )\n        assert branch.parent_id == root.entry_id\n        assert {entry.entry_id for entry in recovered.entries} == {\n            root.entry_id,\n            main.entry_id,\n            branch.entry_id,\n        }\n\n\ndef test_session_factory_forks_from_an_explicit_historical_entry() -> None:\n    async def scenario() -> None:\n        store = MemorySessionStore()\n        state = store.create("factory-fork")\n        with store.writer(state.session_id) as writer:\n            root = writer.append((AgentMessage.text(Role.USER, "root"),))\n            main = writer.append((AgentMessage.text(Role.ASSISTANT, "main"),))\n        session = create_agent_session(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("branch answer"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/fork"),\n            ),\n            store=store,\n            session_id=state.session_id,\n            fork_from=root.entry_id,\n        )\n\n        await session.run("branch question")\n\n        recovered = store.read(state.session_id)\n        assert recovered.history(main.entry_id)[-1] == AgentMessage.text(\n            Role.ASSISTANT, "main"\n        )\n        assert recovered.history() == (\n            AgentMessage.text(Role.USER, "root"),\n            AgentMessage.text(Role.USER, "branch question"),\n            AgentMessage.text(Role.ASSISTANT, "branch answer"),\n        )\n\n    asyncio.run(scenario())\n\n\ndef test_session_factory_creates_new_durable_state_and_no_save_rejects_continuation(\n    tmp_path,\n) -> None:\n    def runtime(answer: str) -> AgentRuntime:\n        return AgentRuntime(\n            ScriptedModelAdapter([TextDelta(answer), ModelEnd(StopReason.COMPLETE)]),\n            ModelSpec("scripted/factory"),\n        )\n\n    store = JSONLSessionStore(tmp_path)\n    first = create_agent_session(runtime("one"), store=store)\n    second = create_agent_session(runtime("two"), store=store)\n\n    assert first.session_id is not None\n    assert second.session_id is not None\n    assert first.session_id != second.session_id\n    with pytest.raises(ValueError, match="no_save.*continuation"):\n        create_agent_session(\n            runtime("never"),\n            store=store,\n            session_id=first.session_id,\n            no_save=True,\n        )\n    ephemeral = create_agent_session(runtime("private"), no_save=True)\n    assert ephemeral.session_id is None\n'

In [ ]:
CHAPTER_CONTRACT_TEST_SOURCE = 'from __future__ import annotations\n\nimport pytest\n\nfrom agent_harness import AgentRuntime, JSONLSessionStore, ModelSpec, ScriptedModelAdapter\nfrom agent_harness import create_agent_session\n\n\ndef test_persistence_inputs_fail_before_session_work_is_accepted(tmp_path) -> None:\n    store = JSONLSessionStore(tmp_path)\n    with pytest.raises(ValueError, match="safe local identifier"):\n        store.create("../escape")\n    with pytest.raises(ValueError, match="continuation requires"):\n        create_agent_session(\n            AgentRuntime(ScriptedModelAdapter([]), ModelSpec("scripted/contract")),\n            session_id="missing-store",\n        )\n'

In [ ]:
with TemporaryDirectory(prefix='chapter-05-tree-') as temporary:
    store = chapter5.JSONLSessionStore(temporary)
    state = store.create('tree-demo')
    with store.writer(state.session_id) as writer:
        root_entry = writer.append((chapter5.AgentMessage.text(chapter5.Role.USER, 'root'),))
        main_entry = writer.append((chapter5.AgentMessage.text(chapter5.Role.ASSISTANT, 'main'),))
        branch_entry = writer.append(
            (chapter5.AgentMessage.text(chapter5.Role.ASSISTANT, 'branch'),),
            parent_id=root_entry.entry_id,
        )
    tree_state = chapter5.JSONLSessionStore(temporary).read('tree-demo')
    assert tree_state.history(main_entry.entry_id)[-1].content[0].text == 'main'
    assert tree_state.history()[-1].content[0].text == 'branch'

## Observable Trace

Session JSONL is resumable state, not a Run Trace. Its observable evidence is deliberately compact: one versioned header followed by append-only settlement records. Entry and parent identifiers expose the tree, while each record's message kinds show which structural boundary was committed.

In [ ]:
observable_trace = [
    {
        'entry_id': entry.entry_id,
        'parent_id': entry.parent_id,
        'message_kinds': [type(message).__name__ for message in entry.messages],
    }
    for entry in minimal_state.entries
]
assert observable_trace[0]['parent_id'] is None
assert observable_trace[1]['parent_id'] == observable_trace[0]['entry_id']
observable_trace

## Failure Boundaries and Trade-offs

Every JSONL settlement is encoded before append, flushed, and synced. A final byte fragment without a newline is reported and ignored; a later writer discards it under the Session Lock before appending. It is never replayed as completed work. The guarantee stops there: version one does not replay a provider stream or a host Tool interrupted before its complete structural settlement.

A Session Lock protects one append target across processes. It is an ownership guarantee, not a stale lock-file protocol: operating-system lock release follows process termination. Read-only access and unrelated Session files do not need that exclusive lease.

In [ ]:
with TemporaryDirectory(prefix='chapter-05-interrupted-') as temporary:
    store = chapter5.JSONLSessionStore(temporary)
    state = store.create('interrupted-demo')
    with store.writer(state.session_id) as writer:
        writer.append((chapter5.AgentMessage.text(chapter5.Role.USER, 'settled'),))
    with store.path_for(state.session_id).open('ab') as stream:
        stream.write(b'{"record":"settlement"')
    recovered = store.read(state.session_id)
    assert recovered.history() == (chapter5.AgentMessage.text(chapter5.Role.USER, 'settled'),)
    assert recovered.recovery_warning.code is chapter5.RecoveryCode.INCOMPLETE_FINAL_RECORD
failure_boundary = {
    'code': recovered.recovery_warning.code.value,
    'line': recovered.recovery_warning.line_number,
}
failure_boundary

## Checkpoint Export and Verification

Chapter metadata names Chapter 4 as the base Checkpoint. Export carries forward unchanged cumulative modules and tests, replaces the evolved files, adds persistence tests, writes a deterministic manifest, then compiles, installs without dependency resolution, imports, and runs every test before publishing Chapter 5.

In [ ]:
PYPROJECT_SOURCE = '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "agent-harness"\nversion = "0.5.0"\ndescription = "Chapter 5 durable tree Sessions and settled recovery"\nreadme = "README.md"\nrequires-python = ">=3.11"\ndependencies = ["jsonschema>=4.23,<5", "openai>=1.40,<3"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.setuptools.package-data]\nagent_harness = ["py.typed"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'

In [ ]:
README_SOURCE = '# Agent Harness — Chapter 5 Checkpoint\n\nThis cumulative Checkpoint turns `AgentSession` into the durable façade around `AgentRuntime`. `MemorySessionStore` and `JSONLSessionStore` implement the same small `SessionStore` interface: create, read, and acquire one writer. Settlements are append-only tree entries with explicit entry and parent identifiers, so continuing from the active leaf or forking a historical entry never flattens prior history.\n\n`AgentRuntime` exposes a Session-owned settlement sink. Complete user or assistant messages commit independently, while an assistant Tool Call and its ordered `ToolResultMessage` values commit as one structural settlement. A process interruption cannot restore an orphan Tool Call or claim mid-stream or mid-Tool replay.\n\n`JSONLSessionStore` writes one versioned file per Session, flushes every settlement, ignores and reports an incomplete final record, and discards that uncommitted tail under the writer lease before continuation. It accepts unknown optional fields within schema major 1 and rejects unsupported majors with migration guidance. `migrate_session_file` writes and validates a separate current-format file without modifying the original.\n\n`create_agent_session` creates new durable state whenever a Store is supplied without a Session id. Continuation and historical forks are explicit. `no_save=True` creates a fully ephemeral Session and rejects persistence or continuation inputs.\n\nDurable writers use cross-platform operating-system file locks. A competing writer fails promptly with structured `session_busy`, while read-only inspection and independent Sessions remain available.\n'

In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '05_durable_sessions.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch05',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '05_durable_sessions.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch05',
) == ()
checkpoint_result

## Public API Summary

Use `MemorySessionStore` or `JSONLSessionStore` through the `SessionStore` interface. `create_agent_session(runtime, store=store)` creates a new durable Session; add `session_id=...` to continue its active leaf or `fork_from=entry_id` to branch from history. `no_save=True` produces an ephemeral Session and rejects continuation.

Read a `SessionState`, inspect `entries` and `recovery_warning`, and call `history()` or `history(entry_id)` to materialize a branch. Competing writers raise `SessionBusyError` with code `session_busy`. Use `migrate_session_file(source, destination)` to produce and validate a new current-format copy while preserving the original.